# SteadyState — steady state with a spatially varying diffusivity

`MagneticStandalone` · ASN 28 validation case (TRUST-LiMe).


## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs **one and only one test case**: ``steady_state``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.

## What it validates

$b=y$ with $\eta(x)=1+x$ and $u=0$. Since $\nabla b=e_y$,
$\nabla\cdot(\eta\nabla b)=\partial_y(1+x)=0$: the profile is an exact steady
solution for any diffusivity that varies only in $x$. Any spurious source or
mis-assembled diffusion operator destroys it, and the diffusivity is read from a
**functional** field, exercising the non-uniform-$\eta$ path.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 6 | diffusion in divergence form $\nabla\cdot(\eta\nabla b)$ | $\eta=1+x$ equilibrium |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
import pathlib, shutil

from trustutils import run

NB_PROCS = 2

run.reset()
case = run.addCase(".", "steady_state.data", nbProcs=NB_PROCS)
case.partition()          # trust -partition steady_state.data 2 -> PAR_steady_state.data + DOM_000*.Zones
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob(f"{case.dataFileName_}_*.son"):
    shutil.copy2(son, build / son.name)


## Checks


In [ ]:
dy_half = 0.5 / 32.0   # 33x33 nodes -> 32 cells over [0,1]; P0 is cell-centered
for tag, y_ref in [("SONDE_Y025", 0.25), ("SONDE_Y050", 0.50), ("SONDE_Y075", 0.75)]:
    t, b = read_son(case.dataFileName_, tag)
    # the equilibrium must not evolve in time
    drift = np.max(np.abs(b - b[0]))
    # the P0 (cell-centered) projection of the linear profile b=y stands within
    # half a cell of the analytic value at every probe
    repr_err = np.abs(b[0] - y_ref)
    print(f"{tag}: b(0)={b[0]:.5f} (analytic {y_ref}), drift={drift:.2e}, P0 repr err={repr_err:.5f}")
    assert drift < 1e-8, f"steady state not preserved at {tag}: drift={drift:.3e}"
    assert repr_err <= dy_half + 1e-6, f"P0 projection off at {tag}: {repr_err:.3e}"
print("OK: equilibrium with non-uniform eta is preserved (P0 projection of b=y).")

import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
t, b = read_son(case.dataFileName_, "SONDE_Y050")
plt.plot(t, b, label="b(x=0.5, y=0.5, t)  [b=y equilibrium]")
plt.axhline(0.5, color="k", ls="--", label="analytic 0.5")
plt.xlabel("t [s]"); plt.ylabel("b"); plt.legend(); plt.grid(True); plt.show()


## Computer Performance


In [ ]:
run.tablePerf()